# Pattern 4: Multi-Agent Supervisor Routing (Claude)

This notebook demonstrates a supervisor that routes work between specialist agents using Claude Haiku 4.5.
It also records tracing spans so you can inspect how the routing decisions unfold.

```mermaid
flowchart TD
    Q[Question] --> S[Supervisor]
    S --> R[Researcher]
    S --> W[Writer]
    R --> S
    W --> S
    S --> F[Finalize]
```

In [1]:
from typing import TypedDict, Literal
import json
from pathlib import Path
from datetime import datetime, timezone

from langchain_anthropic import ChatAnthropic
from langgraph.graph import END, START, StateGraph
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter, SpanExportResult, SpanExporter

MODEL_NAME = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=MODEL_NAME, temperature=0, max_tokens=1024)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
LOG_FILE = Path("trace_logs") / f"04_multi_agent_supervisor_spans_{RUN_TIMESTAMP}.log"


class FileSpanExporter(SpanExporter):
    def __init__(self, log_file: Path):
        self.log_file = log_file
        self.log_file.parent.mkdir(parents=True, exist_ok=True)

    def export(self, spans):
        with self.log_file.open("a", encoding="utf-8") as handle:
            for span in spans:
                record = {
                    "name": span.name,
                    "trace_id": f"{span.context.trace_id:032x}",
                    "span_id": f"{span.context.span_id:016x}",
                    "parent_span_id": f"{span.parent.span_id:016x}" if span.parent else None,
                    "start_time": datetime.fromtimestamp(span.start_time / 1_000_000_000, tz=timezone.utc).isoformat(),
                    "end_time": datetime.fromtimestamp(span.end_time / 1_000_000_000, tz=timezone.utc).isoformat() if span.end_time else None,
                    "attributes": dict(span.attributes),
                    "status": str(span.status.status_code.name),
                }
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        return SpanExportResult.SUCCESS

    def shutdown(self):
        return None

    def force_flush(self, timeout_millis: int = 30000):
        return True


resource = Resource.create({"service.name": "agentic-patterns-notebook-04"})
current_provider = trace.get_tracer_provider()
if hasattr(current_provider, "add_span_processor"):
    provider = current_provider
    if not getattr(provider, "_agentic_patterns_notebook_04_configured", False):
        provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))
        provider.add_span_processor(BatchSpanProcessor(FileSpanExporter(LOG_FILE)))
        setattr(provider, "_agentic_patterns_notebook_04_configured", True)
else:
    provider = TracerProvider(resource=resource)
    provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))
    provider.add_span_processor(BatchSpanProcessor(FileSpanExporter(LOG_FILE)))
    setattr(provider, "_agentic_patterns_notebook_04_configured", True)
    trace.set_tracer_provider(provider)

tracer = trace.get_tracer("pattern-04.multi-agent-supervisor")


def invoke_llm_with_span(prompt: str, node_name: str, pattern_name: str = "multi_agent_supervisor") -> str:
    with tracer.start_as_current_span("llm.invoke") as span:
        span.set_attribute("agent.pattern", pattern_name)
        span.set_attribute("agent.node", node_name)
        span.set_attribute("llm.model", MODEL_NAME)
        span.set_attribute("llm.prompt_length", len(prompt))
        span.set_attribute("llm.prompt", prompt)
        response = llm.invoke(prompt)
        content = str(response.content)
        span.set_attribute("llm.response_length", len(content))
        span.set_attribute("llm.response", content)
        return content

In [ ]:
MAX_WRITER_DRAFTS = 5

class MultiAgentState(TypedDict):
    question: str
    research_notes: str
    draft_answer: str
    final_answer: str
    writer_draft_count: int
    route: Literal['researcher', 'writer', 'finish']


def supervisor_node(state: MultiAgentState) -> MultiAgentState:
    route_prompt = f"""
You are a supervisor. Choose exactly one next route: researcher, writer, or finish.
Question: {state['question']}
Current research notes length: {len(state['research_notes'])}
Current draft length: {len(state['draft_answer'])}
Current writer draft count: {state['writer_draft_count']} / {MAX_WRITER_DRAFTS}

Rules:
- If research notes are weak/empty, choose researcher.
- If research exists but no draft, choose writer.
- If draft exists and is enough, choose finish.
- If writer draft count has reached the max, choose finish.
Return only one word.
"""
    with tracer.start_as_current_span("pattern.supervisor.routing") as pattern_span:
        pattern_span.set_attribute("agent.pattern", "multi_agent_supervisor")
        choice = invoke_llm_with_span(route_prompt, "supervisor").strip().lower()

    # Hard guard: never allow more than MAX_WRITER_DRAFTS writer iterations.
    if state['writer_draft_count'] >= MAX_WRITER_DRAFTS:
        state['route'] = 'finish'
    elif 'researcher' in choice:
        state['route'] = 'researcher'
    elif 'writer' in choice:
        state['route'] = 'writer'
    else:
        state['route'] = 'finish'
    return state


def researcher_node(state: MultiAgentState) -> MultiAgentState:
    prompt = f"""
Collect concise research notes (5 bullets max) for this question:
{state['question']}
"""
    with tracer.start_as_current_span("pattern.researcher.step") as pattern_span:
        pattern_span.set_attribute("agent.pattern", "multi_agent_supervisor")
        state['research_notes'] = invoke_llm_with_span(prompt, "researcher")
    return state


def writer_node(state: MultiAgentState) -> MultiAgentState:
    prompt = f"""
Use these notes to draft a clear answer.
Question: {state['question']}
Notes:
{state['research_notes']}
"""
    with tracer.start_as_current_span("pattern.writer.step") as pattern_span:
        pattern_span.set_attribute("agent.pattern", "multi_agent_supervisor")
        state['draft_answer'] = invoke_llm_with_span(prompt, "writer")
    state['writer_draft_count'] += 1
    return state


def finalize_node(state: MultiAgentState) -> MultiAgentState:
    if state['draft_answer']:
        state['final_answer'] = state['draft_answer']
    else:
        state['final_answer'] = state['research_notes']
    return state


def route_from_supervisor(state: MultiAgentState) -> str:
    return state['route']

In [ ]:
builder = StateGraph(MultiAgentState)
builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('writer', writer_node)
builder.add_node('finalize', finalize_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route_from_supervisor, {
    'researcher': 'researcher',
    'writer': 'writer',
    'finish': 'finalize',
})
builder.add_edge('researcher', 'supervisor')
builder.add_edge('writer', 'supervisor')
builder.add_edge('finalize', END)

graph = builder.compile()

initial = {
    'question': 'What are practical first projects to learn agentic AI on a laptop?',
    'research_notes': '',
    'draft_answer': '',
    'final_answer': '',
    'writer_draft_count': 0,
    'route': 'researcher',
}

result = graph.invoke(initial)
result['final_answer']

{
    "name": "llm.invoke",
    "context": {
        "trace_id": "0xba2444ed8b52c5535fb2127755ce3f45",
        "span_id": "0x18559ffa2505be78",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xcbb38eea5fda7608",
    "start_time": "2026-04-15T18:17:01.014307Z",
    "end_time": "2026-04-15T18:17:01.836759Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "agent.pattern": "multi_agent_supervisor",
        "agent.node": "supervisor",
        "llm.model": "claude-haiku-4-5-20251001",
        "llm.prompt_length": 402,
        "llm.prompt": "\nYou are a supervisor. Choose exactly one next route: researcher, writer, or finish.\nQuestion: What are practical first projects to learn agentic AI on a laptop?\nCurrent research notes length: 0\nCurrent draft length: 0\n\nRules:\n- If research notes are weak/empty, choose researcher.\n- If research exists but no draft, choose writer.\n- If draft exists and is enough, choose fini

KeyboardInterrupt: 

## How the code cells map to the pattern
Cell 2 configures the Claude client, OpenTelemetry exporter, and helper that wraps every model call in a span so the routing decisions are observable.
Cell 3 defines the supervisor state plus the supervisor, researcher, writer, and finalize nodes. This is the actual routing policy of the pattern.
Cell 4 compiles the graph, seeds an example question, and runs the full supervisor loop until the final answer is produced.

## OpenTelemetry in this notebook
This pattern is instrumented with OpenTelemetry so every routing and LLM step is observable.

What gets traced:
- Supervisor routing span: `pattern.supervisor.routing`
- Researcher and writer spans: `pattern.researcher.step`, `pattern.writer.step`
- LLM invocation span for each model call: `llm.invoke`

The tracing setup exports spans in two ways:
- Console exporter for immediate feedback while running cells
- File exporter that appends JSON lines into `trace_logs/`

Each exported record includes IDs, timestamps, status, and attributes like `agent.node` and `llm.model`.
This makes it easier to answer: which node ran, how long it took, and what decision path the supervisor followed.

In [4]:
import json
from pathlib import Path
from datetime import datetime
from IPython.display import HTML, display


def load_latest_supervisor_trace(log_dir: str = "trace_logs"):
    trace_dir = Path(log_dir)
    files = sorted(trace_dir.glob("04_multi_agent_supervisor_spans_*.log"), key=lambda p: p.stat().st_mtime)
    if not files:
        return None, []

    latest = files[-1]
    records = []
    with latest.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                start = datetime.fromisoformat(row["start_time"].replace("Z", "+00:00")) if row.get("start_time") else None
                end = datetime.fromisoformat(row["end_time"].replace("Z", "+00:00")) if row.get("end_time") else None
                row["_start_dt"] = start
                row["_end_dt"] = end
                if start and end:
                    row["duration_ms"] = round((end - start).total_seconds() * 1000, 2)
                else:
                    row["duration_ms"] = None
                attrs = row.get("attributes") or {}
                row["agent_node"] = attrs.get("agent.node", "-")
                row["llm_model"] = attrs.get("llm.model", "-")
                records.append(row)
            except json.JSONDecodeError:
                continue

    records.sort(key=lambda r: r.get("_start_dt") or datetime.min)
    return latest, records


def render_trace_dashboard(records, latest_path):
    if not records:
        display(HTML("""
        <div style='font-family: Segoe UI, Tahoma, sans-serif; background: linear-gradient(135deg,#f7fbff,#eef6ff);
                    border:1px solid #dbe8f5; border-radius:16px; padding:18px;'>
          <h3 style='margin:0 0 8px 0; color:#17324d;'>No trace data found</h3>
          <p style='margin:0; color:#34506b;'>Run the supervisor execution cell first to generate a log under <b>trace_logs/</b>.</p>
        </div>
        """))
        return

    durations = [r["duration_ms"] for r in records if r.get("duration_ms") is not None]
    total_ms = round(sum(durations), 2) if durations else 0.0
    avg_ms = round(total_ms / len(durations), 2) if durations else 0.0
    unique_traces = len({r.get("trace_id") for r in records if r.get("trace_id")})
    span_count = len(records)

    start_points = [r.get("_start_dt") for r in records if r.get("_start_dt")]
    min_start = min(start_points) if start_points else None
    max_end = max([r.get("_end_dt") or r.get("_start_dt") for r in records if r.get("_start_dt")]) if start_points else None
    total_window_ms = max(1.0, (max_end - min_start).total_seconds() * 1000) if min_start and max_end else 1.0

    palette = {
        "supervisor": "#1d4ed8",
        "researcher": "#0ea5e9",
        "writer": "#10b981",
        "-": "#6b7280",
    }

    timeline_rows = []
    for i, r in enumerate(records):
        if not r.get("_start_dt"):
            continue
        left = ((r["_start_dt"] - min_start).total_seconds() * 1000 / total_window_ms) * 100
        width = ((r.get("duration_ms") or 1.0) / total_window_ms) * 100
        width = max(width, 0.6)
        node = r.get("agent_node", "-")
        color = palette.get(node, "#f59e0b")
        timeline_rows.append(
            f"<div style='position:relative; height:26px; margin:6px 0;'>" 
            f"<div style='position:absolute; left:{left:.2f}%; width:{width:.2f}%; height:18px; border-radius:999px; background:{color};'></div>" 
            f"<span style='position:absolute; left:0; top:-2px; font-size:12px; color:#1f2937;'>#{i+1} {r.get('name','-')} ({node})</span>" 
            f"</div>"
        )

    rows_html = []
    for idx, r in enumerate(records, start=1):
        rows_html.append(
            "<tr>"
            f"<td>{idx}</td>"
            f"<td>{r.get('name','-')}</td>"
            f"<td>{r.get('agent_node','-')}</td>"
            f"<td>{r.get('status','-')}</td>"
            f"<td>{r.get('duration_ms','-')}</td>"
            f"<td style='max-width:340px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;'>{r.get('trace_id','-')}</td>"
            "</tr>"
        )

    html = f"""
    <div style='font-family: Segoe UI, Tahoma, sans-serif; color:#0f172a;'>
      <div style='background: radial-gradient(circle at top left, #dbecff, #f6fbff 55%); border:1px solid #d9e7f5;
                  border-radius:18px; padding:18px 20px; box-shadow:0 8px 26px rgba(30,64,175,0.08);'>
        <h2 style='margin:0 0 6px 0; font-size:24px; color:#0b2a4a;'>Notebook 4 Trace Dashboard</h2>
        <p style='margin:0; color:#355069;'>Latest file: <b>{latest_path.name}</b></p>
      </div>

      <div style='display:grid; grid-template-columns:repeat(4,minmax(120px,1fr)); gap:10px; margin:14px 0 18px 0;'>
        <div style='background:#ffffff; border:1px solid #e5edf5; border-radius:14px; padding:10px 12px;'><div style='font-size:12px;color:#64748b;'>Spans</div><div style='font-size:22px;font-weight:700;color:#0f172a;'>{span_count}</div></div>
        <div style='background:#ffffff; border:1px solid #e5edf5; border-radius:14px; padding:10px 12px;'><div style='font-size:12px;color:#64748b;'>Trace IDs</div><div style='font-size:22px;font-weight:700;color:#0f172a;'>{unique_traces}</div></div>
        <div style='background:#ffffff; border:1px solid #e5edf5; border-radius:14px; padding:10px 12px;'><div style='font-size:12px;color:#64748b;'>Total Duration (ms)</div><div style='font-size:22px;font-weight:700;color:#0f172a;'>{total_ms}</div></div>
        <div style='background:#ffffff; border:1px solid #e5edf5; border-radius:14px; padding:10px 12px;'><div style='font-size:12px;color:#64748b;'>Average Span (ms)</div><div style='font-size:22px;font-weight:700;color:#0f172a;'>{avg_ms}</div></div>
      </div>

      <div style='background:#ffffff; border:1px solid #e5edf5; border-radius:14px; padding:14px 16px; margin-bottom:16px;'>
        <h3 style='margin:0 0 8px 0; color:#17324d;'>Timeline</h3>
        <div style='background:#f8fbff; border:1px solid #e5edf5; border-radius:12px; padding:8px 10px;'>
          {''.join(timeline_rows)}
        </div>
      </div>

      <div style='background:#ffffff; border:1px solid #e5edf5; border-radius:14px; padding:14px 16px;'>
        <h3 style='margin:0 0 8px 0; color:#17324d;'>Span Table</h3>
        <table style='border-collapse:collapse; width:100%; font-size:13px;'>
          <thead>
            <tr style='background:#f8fbff; color:#334155;'>
              <th style='text-align:left; padding:8px; border-bottom:1px solid #e5edf5;'>#</th>
              <th style='text-align:left; padding:8px; border-bottom:1px solid #e5edf5;'>Name</th>
              <th style='text-align:left; padding:8px; border-bottom:1px solid #e5edf5;'>Node</th>
              <th style='text-align:left; padding:8px; border-bottom:1px solid #e5edf5;'>Status</th>
              <th style='text-align:left; padding:8px; border-bottom:1px solid #e5edf5;'>Duration (ms)</th>
              <th style='text-align:left; padding:8px; border-bottom:1px solid #e5edf5;'>Trace ID</th>
            </tr>
          </thead>
          <tbody>
            {''.join(rows_html)}
          </tbody>
        </table>
      </div>
    </div>
    """

    display(HTML(html))


latest_log, span_records = load_latest_supervisor_trace()
render_trace_dashboard(span_records, latest_log if latest_log else Path("trace_logs"))

#,Name,Node,Status,Duration (ms),Trace ID
1,pattern.supervisor.routing,-,UNSET,822.56,ba2444ed8b52c5535fb2127755ce3f45
2,llm.invoke,supervisor,UNSET,822.45,ba2444ed8b52c5535fb2127755ce3f45
3,pattern.researcher.step,-,UNSET,4314.13,cca15c6fdc63e8550c5d7340f97c5ecf
4,llm.invoke,researcher,UNSET,4313.98,cca15c6fdc63e8550c5d7340f97c5ecf
5,pattern.supervisor.routing,-,UNSET,634.72,d5f32f27fe5bfc2e1309f6ddad987eed
6,llm.invoke,supervisor,UNSET,634.59,d5f32f27fe5bfc2e1309f6ddad987eed
7,pattern.writer.step,-,UNSET,3807.0,b9dd34e7c57352d4a558eba22b5ad844
8,llm.invoke,writer,UNSET,3806.87,b9dd34e7c57352d4a558eba22b5ad844
9,pattern.supervisor.routing,-,UNSET,516.25,48b142d59a37c093ac8b4b44f761276d
10,llm.invoke,supervisor,UNSET,516.16,48b142d59a37c093ac8b4b44f761276d
